In [1]:
# ============================================================
#  DYNAMIC RISK SCORE SYSTEM — FIXED KAGGLE NOTEBOOK
#  Datasets: Crowded | Hockey Fight | Non-Snatching |
#            Snatching | Shoplifting (normal + shoplifting)
#  Outputs : annotated video, per-frame CSV, summary CSV, .pth
#  Target  : ~98 % accuracy
# ============================================================

# ─── CELL 1 : Install dependencies ───────────────────────────
import subprocess, sys

pkgs = [
    "ultralytics",
    "opencv-python-headless",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "tqdm",
    "Pillow",
]
for p in pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p])

# ─── CELL 2 : Imports ────────────────────────────────────────
import os, glob, random, time, csv, json, math, warnings
from pathlib import Path
from collections import defaultdict, deque
from copy import deepcopy

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix
)

from ultralytics import YOLO

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# ─── CELL 3 : Config & label map ─────────────────────────────
# 0=normal  1=fight  2=snatching  3=shoplifting
DATASET_ROOTS = {
    "crowded"     : ("/kaggle/input/datasets/gauravsingh72509/crowded-dataset",                        0),
    "hockey_fight": ("/kaggle/input/datasets/yassershrief/hockey-fight-vidoes/data",                   1),
    "non_snatch"  : ("/kaggle/input/datasets/gauravsingh72509/non-snatching-datasets/normal",          0),
    "snatching"   : ("/kaggle/input/datasets/gauravsingh72509/snatching-datasets/snatching",           2),
    "shop_normal" : ("/kaggle/input/datasets/kipshidze/shoplifting-video-dataset/normal",              0),
    "shoplifting" : ("/kaggle/input/datasets/kipshidze/shoplifting-video-dataset/shoplifting",         3),
}

CLASS_NAMES = {0: "Normal", 1: "Fight", 2: "Snatching", 3: "Shoplifting"}
RISK_SCORES = {0: 10,       1: 90,      2: 85,          3: 65}   # base 0-100
NUM_CLASSES = 4
CLIP_LEN    = 16
FRAME_SIZE  = 112
VID_EXTS    = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv", ".mpg", ".mpeg"}
OUTPUT_DIR  = Path("/kaggle/working/risk_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── CELL 4 : Collect video paths ────────────────────────────
def collect_videos(roots):
    records = []
    for name, (root, label) in roots.items():
        root = Path(root)
        if not root.exists():
            print(f"  [WARN] missing: {root}")
            continue
        vids = [p for p in root.rglob("*") if p.suffix.lower() in VID_EXTS]
        print(f"  {name:14s}: {len(vids):4d} videos  label={label}")
        for v in vids:
            records.append({"path": str(v), "label": label,
                            "dataset": name, "filename": v.name})
    return pd.DataFrame(records)

print("Scanning datasets …")
df_all = collect_videos(DATASET_ROOTS)
print(f"\nTotal videos: {len(df_all)}")
print(df_all["label"].value_counts())

# ─── CELL 5 : Transforms & Dataset ───────────────────────────
transform_train = T.Compose([
    T.RandomResizedCrop(FRAME_SIZE, scale=(0.75, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize([0.45, 0.45, 0.45], [0.225, 0.225, 0.225]),
])
transform_val = T.Compose([
    T.Resize((FRAME_SIZE, FRAME_SIZE)),
    T.ToTensor(),
    T.Normalize([0.45, 0.45, 0.45], [0.225, 0.225, 0.225]),
])


def read_clip(path, augment=False):
    """Read CLIP_LEN evenly-spaced frames → tensor (3, T, H, W)."""
    cap   = cv2.VideoCapture(str(path))
    total = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 1)
    idxs  = np.linspace(0, total - 1, CLIP_LEN, dtype=int)
    tfm   = transform_train if augment else transform_val
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ret, fr = cap.read()
        if not ret:
            fr = np.zeros((FRAME_SIZE, FRAME_SIZE, 3), np.uint8)
        else:
            fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        frames.append(tfm(Image.fromarray(fr)))
    cap.release()
    return torch.stack(frames, dim=1)   # (3, T, H, W)


class VideoDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):  return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row["label"])
        try:
            clip = read_clip(row["path"], augment=self.augment)
        except Exception:
            clip = torch.zeros(3, CLIP_LEN, FRAME_SIZE, FRAME_SIZE)
        return clip, label

# ─── CELL 6 : Pure-3D Model (no 2-D BN anywhere) ─────────────
# Every layer is built from scratch using nn.Conv3d + nn.BatchNorm3d.
# No inflation of pretrained 2-D weights = zero BatchNorm dimension mismatch.

class BasicBlock3D(nn.Module):
    """3-D analogue of ResNet BasicBlock."""
    expansion = 1

    def __init__(self, in_ch, out_ch, stride=(1,1,1), downsample=None):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch,
                               kernel_size=(3,3,3), stride=stride,
                               padding=(1,1,1), bias=False)
        self.bn1   = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch,
                               kernel_size=(3,3,3), stride=(1,1,1),
                               padding=(1,1,1), bias=False)
        self.bn2   = nn.BatchNorm3d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample:
            identity = self.downsample(x)
        return self.relu(out + identity)


def make_layer3d(in_ch, out_ch, blocks, stride=(1,2,2)):
    downsample = None
    if stride != (1,1,1) or in_ch != out_ch:
        downsample = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm3d(out_ch),
        )
    layers = [BasicBlock3D(in_ch, out_ch, stride=stride, downsample=downsample)]
    for _ in range(1, blocks):
        layers.append(BasicBlock3D(out_ch, out_ch))
    return nn.Sequential(*layers)


class TemporalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim // 4)
        self.fc2 = nn.Linear(dim // 4, 1)

    def forward(self, x):          # x: (B, T, D)
        w = torch.softmax(self.fc2(F.relu(self.fc1(x))), dim=1)  # (B, T, 1)
        return (w * x).sum(dim=1)                                  # (B, D)


class RiskHead(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.cls = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(0.45),
            nn.Linear(in_dim, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
        self.risk_reg = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Dropout(0.3),
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.cls(x), self.risk_reg(x).squeeze(-1) * 100.0


class DynamicRiskModel(nn.Module):
    """
    Pure-3D ResNet-18 backbone (all BatchNorm3d) +
    Temporal Attention + dual head (class logits + risk score).
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        # stem
        self.stem = nn.Sequential(
            nn.Conv3d(3, 64, kernel_size=(3,7,7),
                      stride=(1,2,2), padding=(1,3,3), bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1,3,3), stride=(1,2,2), padding=(0,1,1)),
        )
        # residual layers  (spatial stride 2 at layer2-4, temporal stride 1)
        self.layer1 = make_layer3d( 64,  64, 2, stride=(1,1,1))
        self.layer2 = make_layer3d( 64, 128, 2, stride=(1,2,2))
        self.layer3 = make_layer3d(128, 256, 2, stride=(1,2,2))
        self.layer4 = make_layer3d(256, 512, 2, stride=(1,2,2))

        self.avgpool  = nn.AdaptiveAvgPool3d((None, 1, 1))   # keep T dim
        self.temporal = TemporalAttention(512)
        self.head     = RiskHead(512, num_classes)

        # weight init
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):                     # x: (B, 3, T, H, W)
        x = self.stem(x)                      # → (B, 64, T, 28, 28)
        x = self.layer1(x)                    # → (B, 64,  T, 28, 28)
        x = self.layer2(x)                    # → (B,128,  T, 14, 14)
        x = self.layer3(x)                    # → (B,256,  T,  7,  7)
        x = self.layer4(x)                    # → (B,512,  T,  4,  4)
        x = self.avgpool(x)                   # → (B,512,  T,  1,  1)
        x = x.squeeze(-1).squeeze(-1)         # → (B,512,  T)
        x = x.permute(0, 2, 1)               # → (B,  T, 512)
        x = self.temporal(x)                  # → (B, 512)
        return self.head(x)


# ── sanity check ─────────────────────────────────────────────
model = DynamicRiskModel(NUM_CLASSES).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
with torch.no_grad():
    _t          = torch.randn(2, 3, CLIP_LEN, FRAME_SIZE, FRAME_SIZE).to(DEVICE)
    _lg, _rs    = model(_t)
print(f"✓ logits {_lg.shape}  risk_score {_rs.shape}")
del _t, _lg, _rs

# ─── CELL 7 : Train / Val split ──────────────────────────────
assert len(df_all) > 0, "No videos found — check dataset paths!"

df_train, df_val = train_test_split(
    df_all, test_size=0.15,
    stratify=df_all["label"], random_state=42
)
print(f"Train: {len(df_train)}  Val: {len(df_val)}")

ds_train = VideoDataset(df_train, augment=True)
ds_val   = VideoDataset(df_val,   augment=False)

# Weighted sampler to handle class imbalance
label_counts = df_train["label"].value_counts().to_dict()
weights      = [1.0 / label_counts[l] for l in df_train["label"]]
sampler      = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

dl_train = DataLoader(ds_train, batch_size=8, sampler=sampler,
                      num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=8, shuffle=False,
                      num_workers=2, pin_memory=True)

# ─── CELL 8 : Loss ───────────────────────────────────────────
class CombinedLoss(nn.Module):
    def __init__(self, label_smooth=0.1, risk_w=0.08):
        super().__init__()
        self.ce     = nn.CrossEntropyLoss(label_smoothing=label_smooth)
        self.risk_w = risk_w

    def forward(self, logits, risk_sc, targets):
        cls_loss  = self.ce(logits, targets)
        target_rs = torch.tensor(
            [RISK_SCORES[t.item()] / 100.0 for t in targets],
            device=targets.device, dtype=torch.float32
        )
        risk_loss = F.mse_loss(risk_sc / 100.0, target_rs)
        return cls_loss + self.risk_w * risk_loss

criterion = CombinedLoss()

# ─── CELL 9 : Training ───────────────────────────────────────
EPOCHS    = 30
LR        = 3e-4
BEST_PATH = OUTPUT_DIR / "risk_model_best.pth"

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_acc  = 0.0
history       = []

for epoch in range(1, EPOCHS + 1):
    # ── train ─────────────────────────────────────────────────
    model.train()
    t_loss = t_correct = t_total = 0
    for clips, labels in tqdm(dl_train, desc=f"Ep{epoch:02d}/train", leave=False):
        clips, labels = clips.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits, risk_sc = model(clips)
        loss = criterion(logits, risk_sc, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss    += loss.item() * clips.size(0)
        t_correct += (logits.argmax(1) == labels).sum().item()
        t_total   += clips.size(0)
    scheduler.step()

    # ── val ───────────────────────────────────────────────────
    model.eval()
    v_loss = v_correct = v_total = 0
    v_preds, v_labels = [], []
    with torch.no_grad():
        for clips, labels in tqdm(dl_val, desc=f"Ep{epoch:02d}/val", leave=False):
            clips, labels = clips.to(DEVICE), labels.to(DEVICE)
            logits, risk_sc = model(clips)
            loss  = criterion(logits, risk_sc, labels)
            preds = logits.argmax(1)
            v_loss    += loss.item() * clips.size(0)
            v_correct += (preds == labels).sum().item()
            v_total   += clips.size(0)
            v_preds.extend(preds.cpu().numpy())
            v_labels.extend(labels.cpu().numpy())

    tr_acc  = t_correct / t_total
    val_acc = v_correct / v_total
    history.append({"epoch": epoch,
                    "train_loss": round(t_loss/t_total, 5),
                    "train_acc" : round(tr_acc, 5),
                    "val_loss"  : round(v_loss/v_total, 5),
                    "val_acc"   : round(val_acc, 5)})
    flag = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch"      : epoch,
            "state_dict" : model.state_dict(),
            "optimizer"  : optimizer.state_dict(),
            "val_acc"    : val_acc,
            "num_classes": NUM_CLASSES,
            "class_names": CLASS_NAMES,
            "frame_size" : FRAME_SIZE,
            "clip_len"   : CLIP_LEN,
        }, BEST_PATH)
        flag = "  ✓ saved"
    print(f"Ep {epoch:02d}  tr_acc={tr_acc:.4f}  val_acc={val_acc:.4f}{flag}")

pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print(f"\nBest val accuracy: {best_val_acc*100:.2f}%")

# ─── CELL 10 : Final evaluation ──────────────────────────────
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.eval()

fp, fl = [], []
with torch.no_grad():
    for clips, labels in tqdm(dl_val, desc="Eval"):
        clips = clips.to(DEVICE)
        logits, _ = model(clips)
        fp.extend(logits.argmax(1).cpu().numpy())
        fl.extend(labels.numpy())

print(f"\nFinal Accuracy : {accuracy_score(fl, fp)*100:.2f}%")
print(classification_report(fl, fp, target_names=list(CLASS_NAMES.values())))

# ─── CELL 11 : YOLO person detector ──────────────────────────
yolo = YOLO("yolov8n.pt")

# ─── CELL 12 : Per-person risk tracker ───────────────────────
class PersonRiskTracker:
    def __init__(self, pid, alpha=0.25, max_hist=60):
        self.pid          = pid
        self.alpha        = alpha
        self.score        = 10.0
        self.history      = deque(maxlen=max_hist)
        self.boxes        = deque(maxlen=10)
        self.label        = 0
        self.level        = "LOW"
        self.frames_seen  = 0

    def update(self, class_probs, raw_risk, bbox):
        weighted     = sum(RISK_SCORES[c] * float(class_probs[c])
                           for c in range(NUM_CLASSES))
        motion_bonus = 0.0
        if len(self.boxes) >= 2:
            pb = self.boxes[-1]
            cx0, cy0 = (pb[0]+pb[2])/2, (pb[1]+pb[3])/2
            cx1, cy1 = (bbox[0]+bbox[2])/2, (bbox[1]+bbox[3])/2
            motion_bonus = min(math.hypot(cx1-cx0, cy1-cy0) / 5.0, 15.0)

        combined    = 0.6*weighted + 0.3*raw_risk + 0.1*motion_bonus
        combined    = float(np.clip(combined, 0, 100))
        self.score  = self.alpha * combined + (1-self.alpha) * self.score
        self.score  = float(np.clip(self.score, 0, 100))
        self.label  = int(np.argmax(class_probs))
        self.level  = self._to_level(self.score)
        self.history.append(self.score)
        self.boxes.append(bbox)
        self.frames_seen += 1

    @staticmethod
    def _to_level(s):
        if s >= 70: return "HIGH"
        if s >= 40: return "MEDIUM"
        return "LOW"

    @property
    def trend(self):
        arr = list(self.history)
        if len(arr) < 5: return "STABLE"
        slope = np.polyfit(range(len(arr[-10:])), arr[-10:], 1)[0]
        if slope >  1.5: return "RISING"
        if slope < -1.5: return "FALLING"
        return "STABLE"


_COLORS = {"LOW": (0,200,0), "MEDIUM": (0,165,255), "HIGH": (0,0,255)}

# ─── CELL 13 : Video inference pipeline ──────────────────────
def infer_video(video_path, label_hint=None, max_frames=None):
    vpath    = Path(video_path)
    out_vid  = OUTPUT_DIR / f"{vpath.stem}_annotated.mp4"
    out_csv  = OUTPUT_DIR / f"{vpath.stem}_frame_risk.csv"

    cap     = cv2.VideoCapture(str(vpath))
    fps     = cap.get(cv2.CAP_PROP_FPS) or 25.0
    W, H    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer  = cv2.VideoWriter(str(out_vid),
                               cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    trackers     = {}
    next_id      = 0
    frame_rows   = []
    clip_buf     = deque(maxlen=CLIP_LEN)
    last_probs   = np.array([1.0, 0.0, 0.0, 0.0])
    last_risk    = 10.0
    frame_idx    = 0

    model.eval()
    with torch.no_grad():
        while True:
            ret, frame = cap.read()
            if not ret or (max_frames and frame_idx >= max_frames):
                break
            ts  = frame_idx / fps
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # ── clip classification ──────────────────────────
            clip_buf.append(transform_val(Image.fromarray(rgb)))
            if len(clip_buf) == CLIP_LEN:
                clip_t = torch.stack(list(clip_buf), dim=1).unsqueeze(0).to(DEVICE)
                logits, risk_sc = model(clip_t)
                last_probs = F.softmax(logits, -1).squeeze(0).cpu().numpy()
                last_risk  = float(risk_sc.item())

            # ── person detection ─────────────────────────────
            res   = yolo(frame, classes=[0], verbose=False)[0]
            boxes = res.boxes.xyxy.cpu().numpy() if res.boxes else np.empty((0,4))

            assigned = {}
            for box in boxes:
                x1,y1,x2,y2 = map(int, box)
                cx,cy = (x1+x2)//2, (y1+y2)//2
                best_id, best_d = None, 80
                for pid, tr in trackers.items():
                    if not tr.boxes: continue
                    pb = tr.boxes[-1]
                    d  = math.hypot((pb[0]+pb[2])/2-cx, (pb[1]+pb[3])/2-cy)
                    if d < best_d: best_d=d; best_id=pid
                if best_id is None:
                    best_id = next_id; next_id += 1
                    trackers[best_id] = PersonRiskTracker(best_id)
                assigned[best_id] = (x1,y1,x2,y2)
                trackers[best_id].update(last_probs, last_risk, (x1,y1,x2,y2))

            # ── annotate ─────────────────────────────────────
            ann         = frame.copy()
            scene_cls   = CLASS_NAMES[int(np.argmax(last_probs))]
            scene_lv    = PersonRiskTracker._to_level(last_risk)
            sc_col      = _COLORS[scene_lv]

            cv2.rectangle(ann, (0,0), (380,68), (20,20,20), -1)
            cv2.putText(ann, f"SCENE: {scene_cls}  [{scene_lv}]",
                        (8,24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, sc_col, 2)
            cv2.putText(ann, f"Risk:{last_risk:.1f}/100   t={ts:.2f}s",
                        (8,52), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

            for pid, (x1,y1,x2,y2) in assigned.items():
                tr  = trackers[pid]
                col = _COLORS[tr.level]
                cv2.rectangle(ann, (x1,y1), (x2,y2), col, 2)
                cv2.putText(ann, f"P{pid} {tr.score:.0f} {tr.level}",
                            (x1, max(y1-6,0)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 2)
                # vertical risk bar
                bar_h = y2-y1; filled = int(bar_h * tr.score / 100)
                cv2.rectangle(ann, (x2+4, y2-filled), (x2+12, y2), col, -1)
                cv2.rectangle(ann, (x2+4, y1), (x2+12, y2), (160,160,160), 1)

                frame_rows.append({
                    "frame_idx"          : frame_idx,
                    "timestamp_sec"      : round(ts, 4),
                    "person_id"          : pid,
                    "person_risk_score"  : round(tr.score, 2),
                    "person_risk_level"  : tr.level,
                    "risk_trend"         : tr.trend,
                    "pred_class"         : CLASS_NAMES[tr.label],
                    "prob_normal"        : round(float(last_probs[0]), 4),
                    "prob_fight"         : round(float(last_probs[1]), 4),
                    "prob_snatching"     : round(float(last_probs[2]), 4),
                    "prob_shoplifting"   : round(float(last_probs[3]), 4),
                    "scene_risk_score"   : round(last_risk, 2),
                    "scene_risk_level"   : scene_lv,
                    "scene_class"        : scene_cls,
                    "bbox_x1"            : x1, "bbox_y1": y1,
                    "bbox_x2"            : x2, "bbox_y2": y2,
                    "bbox_width"         : x2-x1,
                    "bbox_height"        : y2-y1,
                    "frames_tracked"     : tr.frames_seen,
                    "video_name"         : vpath.name,
                    "dataset_label"      : label_hint if label_hint is not None else -1,
                })

            writer.write(ann)
            frame_idx += 1

    cap.release(); writer.release()

    # ── summary dict ─────────────────────────────────────────
    s = {"video_name": vpath.name, "dataset_label": label_hint,
         "total_frames": frame_idx, "fps": round(fps,2),
         "duration_sec": round(frame_idx/fps,2),
         "width": W, "height": H,
         "num_persons_detected": len(trackers),
         "final_scene_class": scene_cls,
         "final_scene_risk": round(last_risk,2),
         "final_scene_level": scene_lv,
         "annotated_video_path": str(out_vid)}

    if frame_rows:
        df_fr = pd.DataFrame(frame_rows)
        df_fr.to_csv(str(out_csv), index=False)
        s["frame_csv_path"]     = str(out_csv)
        s["max_person_risk"]    = round(df_fr["person_risk_score"].max(), 2)
        s["mean_person_risk"]   = round(df_fr["person_risk_score"].mean(), 2)
        s["high_risk_frames"]   = int((df_fr["person_risk_level"]=="HIGH").sum())
        s["medium_risk_frames"] = int((df_fr["person_risk_level"]=="MEDIUM").sum())
        s["low_risk_frames"]    = int((df_fr["person_risk_level"]=="LOW").sum())
        s["dominant_class"]     = df_fr["pred_class"].mode()[0]
        s["risk_rising_frames"] = int((df_fr["risk_trend"]=="RISING").sum())
        pk = df_fr.loc[df_fr["person_risk_score"].idxmax()]
        s["risk_peak_time_sec"] = round(float(pk["timestamp_sec"]), 4)
        s["risk_peak_person_id"]= int(pk["person_id"])
    else:
        s.update({"frame_csv_path":"","max_person_risk":0,"mean_person_risk":0,
                  "high_risk_frames":0,"medium_risk_frames":0,"low_risk_frames":0,
                  "dominant_class":"Unknown","risk_rising_frames":0,
                  "risk_peak_time_sec":0,"risk_peak_person_id":-1})
    return str(out_vid), str(out_csv), s

# ─── CELL 14 : Run inference on sampled videos ───────────────
MAX_PER_LABEL = 10   # ↑ increase if runtime allows

sample_rows = []
for lbl in sorted(df_all["label"].unique()):
    sub = df_all[df_all["label"] == lbl]
    sample_rows.append(sub.head(MAX_PER_LABEL))
df_sample = pd.concat(sample_rows).reset_index(drop=True)
print(f"Inference on {len(df_sample)} videos …")

all_summaries = []
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    try:
        _, _, summ = infer_video(row["path"],
                                 label_hint=int(row["label"]),
                                 max_frames=300)
        all_summaries.append(summ)
    except Exception as e:
        print(f"  [SKIP] {row['path']} → {e}")

# ─── CELL 15 : Master summary CSV ────────────────────────────
SUMMARY_CSV = OUTPUT_DIR / "summary_all_videos.csv"
df_sum = pd.DataFrame(all_summaries)
df_sum.to_csv(str(SUMMARY_CSV), index=False)
print(f"Summary CSV → {SUMMARY_CSV}")
print(df_sum.head())

# ─── CELL 16 : Master frame CSV ──────────────────────────────
frame_csvs = list(OUTPUT_DIR.glob("*_frame_risk.csv"))
if frame_csvs:
    df_all_frames = pd.concat([pd.read_csv(p) for p in frame_csvs], ignore_index=True)
    MASTER_CSV = OUTPUT_DIR / "all_frames_risk_progression.csv"
    df_all_frames.to_csv(str(MASTER_CSV), index=False)
    print(f"Master frame CSV → {MASTER_CSV}")
    print(f"Columns : {list(df_all_frames.columns)}")
    print(df_all_frames.head(3))

# ─── CELL 17 : Final report ───────────────────────────────────
print("\n" + "="*60)
print("  DYNAMIC RISK SCORE SYSTEM — COMPLETE")
print("="*60)
print(f"  Best accuracy    : {best_val_acc*100:.2f}%")
print(f"  Model (.pth)     : {BEST_PATH}")
print(f"  Summary CSV      : {SUMMARY_CSV}")
print(f"  Annotated videos : {len(list(OUTPUT_DIR.glob('*_annotated.mp4')))}")
if frame_csvs:
    print(f"  Master frame CSV : {MASTER_CSV}")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device : cuda
PyTorch: 2.10.0+cu128
Scanning datasets …
  crowded       :   28 videos  label=0
  hockey_fight  : 1000 videos  label=1
  non_snatch    :   82 videos  label=0
  snatching     :   74 videos  label=2
  shop_normal   :   90 videos  label=0
  shoplifting   :   92 videos  label=3

Total videos: 1366
label
1    1000
0     200
3      92
2      74
Name: count, dtype: int64
Parameters: 33,399,366
✓ logits torch.Size([2, 4])  risk_score torch.Size([2])
Train: 1161  Val: 205


Ep 01  tr_acc=0.6494  val_acc=0.8585  ✓ saved


Ep 02  tr_acc=0.7356  val_acc=0.8878  ✓ saved


Ep 03  tr_acc=0.7778  val_acc=0.9024  ✓ saved


Ep 04  tr_acc=0.7838  val_acc=0.8634


...

Ep 30  tr_acc=0.9793  val_acc=0.9463

Best val accuracy: 95.61%


Eval: 100%|██████████| 26/26 [00:49<00:00,  1.90s/it]



Final Accuracy : 95.61%
              precision    recall  f1-score   support

      Normal       0.90      0.87      0.88        30
       Fight       0.99      1.00      1.00       150
   Snatching       0.73      0.73      0.73        11
 Shoplifting       0.86      0.86      0.86        14

    accuracy                           0.96       205
   macro avg       0.87      0.86      0.87       205
weighted avg       0.96      0.96      0.96       205

Inference on 40 videos …


100%|██████████| 40/40 [04:56<00:00,  7.41s/it]


Summary CSV → /kaggle/working/risk_output/summary_all_videos.csv
                               video_name  dataset_label  total_frames   fps  \
0  Screen Recording 2026-04-01 201643.mp4              0           300  30.0   
1  Screen Recording 2026-04-01 204627.mp4              0           300  30.0   
2  Screen Recording 2026-04-01 203549.mp4              0           214  30.0   
3  Screen Recording 2026-04-01 203649.mp4              0           300  30.0   
4  Screen Recording 2026-04-01 203750.mp4              0           300  30.0   

   duration_sec  width  height  num_persons_detected final_scene_class  \
0         10.00   1034     506                    20         Snatching   
1         10.00   1046     512                    15         Snatching   
2          7.13   1036     536                    12         Snatching   
3         10.00   1032     514                    21         Snatching   
4         10.00   1036     552                    13            Normal   

   final_